# colab_18 — scGPT CPT eval #3 / detector #2 (catastrophic forgetting)

Did the aggregated scGPT continued-pretraining run (colab_16) **overwrite the base model's general
cell-type knowledge** while adapting to AD glia? colab_17 asked whether CPT *helped* the biology
(substate, APOE) and found it did not; this notebook asks whether it *hurt* what the model already
knew. It is the last piece of the scGPT arm's verdict.

**Eval #3 and detector #2 are one measurement read two ways** (`docs/EVALUATION_CONTRACT.md`): k-NN
cell-type accuracy on a non-AD reference, frozen base vs post-CPT. The *drop* is scored against eval
#3's reporting bands (ca. 5% acceptable / 5-15% concerning / >15% catastrophic) **and** detector #2's
hard gate (>20% drop = overtrain).

**Why this is a live question here, not a formality.** The Geneformer arm's forgetting probe
(colab_13) came back at +0.09 / +0.24 pp — no detectable forgetting — but that checkpoint's drift was
modest (14.0% / 11.5% of the within-donor substate reference). colab_16's scGPT drift is a different
regime: 22.25x its measured noise floor, and 234.5% / 149.3% (micro / astro) of the substate
reference recomputed in scGPT's own space. The embedding moved *further than the distance between
biologically distinct cell states*, and colab_17 showed that movement does not land on either scored
axis. If it also has not damaged general cell-type knowledge, then the drift is orthogonal to
everything this project can currently name — which is a real result about what CPT does, not a gap in
the measurement.

**Same reference, same protocol, same cells as the Geneformer arm.** The probe reuses colab_13's
non-AD PBMC reference (Kang et al. 2018), its per-type cap, its seed, its donor-disjoint split and
its k-NN settings, and §2b hard-stops if the resulting cell set does not reproduce colab_13's
recorded 13,738 cells / 8 types. Without that, a cross-FM difference could be a difference in which
cells were scored rather than in the models. PBMC's known limitation carries over unchanged and
applies to both arms equally: microglia are myeloid, so the monocyte/DC compartment overlaps the
lineage CPT trained on, and PBMC can therefore *under-detect* forgetting there — the lymphoid types
(T/B/NK) are the genuinely out-of-domain part carrying the probe.

**One extraction point, and why that is complete here.** colab_13 had to read Geneformer at both
`emb_layer=-1` and `emb_layer=0` because that arm's LoRA gain sat downstream of the pipeline readout,
so damage could hide at one depth. scGPT has no such gap: the cell embedding is the `<cls>` position
at the top of the encoder, every LoRA target is upstream of it, and the `ExprDecoder` is frozen and
downstream. The single readout sees the complete adapted representation. It does not report on
whether intermediate layers were degraded in ways `<cls>` reabsorbs — that is a different question
and this notebook does not answer it.

**What is new relative to colab_13: the metric's noise floor is measured, not assumed.** Geneformer's
embedding is deterministic, so colab_13 needed one base pass and one CPT pass. scGPT randomly
subsamples the genes of any cell above its 1,200-token context on *every* pass, so a single-pass drop
cannot be told apart from re-embedding stochasticity by inspection. Here the frozen base and the
merged adapter are each embedded five times, the k-NN is run on every pass, and the drop is read
against a null built from within-arm pass pairs (where the true effect is zero by construction). This
is affordable because the capped reference is ca. one tenth of the glia substrate — ten passes cost a
few minutes, so there is no reason to gate it behind a flag as colab_17 had to.

**A stochasticity asymmetry this notebook has to watch.** colab_17 found that CPT roughly *halved*
the embedding's sensitivity to that random gene subsampling (0.00363 pre-CPT vs 0.0015 post-merge on
the same cells). If that carries to the reference, the base arm is measured under more embedding
noise than the CPT arm, which biases its k-NN accuracy *downward* and therefore makes the drop look
*smaller* — i.e. it makes detector #2 anti-conservative, in the one direction that matters for a
safety gate. §5c measures the per-arm spread directly, restricted to the cells where that spread is
even a meaningful quantity (see below), and reads the probe a second time on the subset of cells that
sit **under** the context limit, where both arms are deterministic and the asymmetry cannot exist.
§6a promotes that second read to operative for the gate only when both the asymmetry is actually
observed and the subset covers the same classification problem as the primary read — otherwise the
primary number stays operative, and it remains the one reported for cross-FM comparability either way.

**Why the stability measurement itself needs a cell population, not just any cells.** Cells under the
context limit embed identically on every pass by construction — repeat "differences" there are pure
float32 summation noise, randomly signed. PBMC cells carry far fewer detected genes than the glia
substrate (which was 86.3% over-context), so this reference could easily have a *minority* of
over-context cells. If the per-arm stability statistic were computed over all cells rather than just
the over-context ones, it would be dominated by that noise whenever the over-context cells are the
minority, making the ratio meaningless rather than merely noisy — this is why §5c restricts it to
`OVER_MASK` and requires a minimum count before trusting it.

**Contract note.** Measuring a null for a pre-registered metric is not the same move as retuning the
band to fit it. Every band and the >20% gate are used exactly as `docs/EVALUATION_CONTRACT.md`
records them; the measured null is reported alongside, never substituted for them.

**Runtime.** GPU required throughout (there is no CPU-only tier — no reference embedding exists on
Drive yet). Restart the runtime after §1a, **then re-run §1a** (the restart clears all Python state,
including the flags, paths and imports 1a itself defines) before continuing to §1b onward.

## 1 — Setup

### 1a — Run-control flags, Drive, repo, scGPT install, checkpoint

One live switch. `SMOKE` thins the reference to a handful of cells per (cell type x donor) group so
the whole path — install, vocabulary mapping, ten embedding passes, split, k-NN — can be rehearsed
cheaply; it suffixes every write and never touches the audit trace. Everything else is fixed: the
pass counts, the seed and the per-type cap are inherited from colab_13 so the two FM arms score the
same cells.

The scGPT install is the same `--no-deps` source install at the same pinned commit every existing
scGPT embedding in this project was produced under, with flash-attn deliberately absent so the
attention path matches colab_10 / colab_16 / colab_17.

**Restart the runtime after this cell, then re-run this cell** before continuing to 1b — the restart
clears every name this cell defines (`SMOKE`, `REPO_PATH`, `TODAY`, `SUFFIX`, `SCGPT_COMMIT`,
`MODEL_DIR`, the `os`/`torch` imports, ...), and 1b onward assumes they exist.

In [ ]:
import os, subprocess, sys
from google.colab import drive
from datetime import date

# ------------------------------ the one live switch ------------------------------
SMOKE = False   # plumbing rehearsal on a tiny subsample; never writes the audit trace
# ---------------------------------------------------------------------------------

# Pass counts: five each, against colab_17's three. Each pass there covered the full 142,588-cell
# glia substrate; the capped reference here is ca. 13.7k cells, so a pass costs roughly a tenth as
# much and a tighter null is close to free.
N_BASE_PASSES = 5   # frozen base -> the null on the k-NN metric (true effect is zero by construction)
N_CPT_PASSES  = 5   # merged adapter -> spread on the real effect

# Inherited from colab_13 verbatim so the two FM arms score the identical cell set.
SEED              = 0
CAP_PER_TYPE      = 3000
SMOKE_N_PER_GROUP = 25

SUFFIX  = "_SMOKE" if SMOKE else ""
RUN_TAG = "seed0"          # the colab_16 checkpoint this probe scores
TODAY   = date.today().isoformat()

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print("Python:", sys.version.split()[0])
print("repo commit:", subprocess.run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"],
                                     capture_output=True, text=True).stdout.strip())

SCGPT_PIN  = "cebd6fae655b9c585a4807daa3ac31bb764f06b4"
MODEL_DIR  = os.path.join(DRIVE_ROOT, "scgpt_whole_human")
CKPT_FILES = ["vocab.json", "args.json", "best_model.pt"]

# scGPT source only (--no-deps) at the commit every existing scGPT embedding was produced under.
# flash-attn stays absent -> PyTorch attention, the path colab_10/16/17 all embedded under.
!pip install --no-deps "git+https://github.com/bowang-lab/scGPT.git@{SCGPT_PIN}"
!pip install -r {REPO_PATH}/requirements_scgpt.txt
# Colab's base image ships torchao 0.10.0, and peft 0.19.1's `is_torchao_available()` RAISES on any
# version below 0.16.0 instead of returning False -- that kills the adapter reload in 4b. Nothing in
# the scGPT stack imports torchao, so removing it makes the probe return False.
!pip uninstall -y torchao

# 4b reloads colab_16's adapter with PeftModel.from_pretrained + merge_and_unload. peft's
# nn.MultiheadAttention merge behaviour is correctness-critical (docs/ASSUMPTIONS.md) -- assert the
# version rather than assume it.
import peft
PEFT_PIN = "0.19.1"
assert peft.__version__ == PEFT_PIN, (
    f"peft {peft.__version__} != pinned {PEFT_PIN}; the adapter reload/merge in 4b depends on its "
    "nn.MultiheadAttention behaviour -- do not proceed on a different version.")
print("peft:", peft.__version__)

def _have_ckpt(d):
    return all(os.path.exists(os.path.join(d, f)) for f in CKPT_FILES)

if not _have_ckpt(MODEL_DIR):
    os.makedirs(MODEL_DIR, exist_ok=True)
    WHOLE_HUMAN_FOLDER = "1oWh_-ZRdhtoGQ2Fw24HP41FgLoomVo-y"   # scGPT README pretrained table
    !pip install -q gdown
    import gdown
    gdown.download_folder(id=WHOLE_HUMAN_FOLDER, output=MODEL_DIR, quiet=False, use_cookies=False)
    hits = [dp for dp, _, fs in os.walk(MODEL_DIR) if "best_model.pt" in fs]
    assert hits, f"best_model.pt not found under {MODEL_DIR} after download"
    MODEL_DIR = hits[0]
assert _have_ckpt(MODEL_DIR), f"checkpoint incomplete in {MODEL_DIR}: need {CKPT_FILES}"

import torch
assert torch.cuda.is_available(), (
    "this notebook needs a GPU runtime throughout -- there is no CPU-only tier, because no "
    "reference embedding exists on Drive to read from.")
SCGPT_COMMIT = SCGPT_PIN
print("scGPT commit:", SCGPT_COMMIT[:7], "| checkpoint:", MODEL_DIR,
      "| GPU:", torch.cuda.get_device_name(0))
print(f"\nSMOKE={SMOKE} | passes base/cpt = {N_BASE_PASSES}/{N_CPT_PASSES} | suffix={SUFFIX!r}")
print("RESTART THE RUNTIME NOW, then RE-RUN THIS CELL (1a) before continuing from 1b onward.")

### 1b — pip freeze + env JSON (records the exact probe-run stack)

The same two-artifact snapshot every notebook in this project writes: a full `pip freeze` and a JSON
with the interpreter, GPU, CUDA, repo commit and the pinned scGPT and peft versions. Pin files
declare intent; these record what actually resolved at run time, which is what a Methods section
needs.

In [ ]:
import json, platform, subprocess, sys

NOTEBOOK_ID  = "colab_18"
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze{SUFFIX}.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        m = __import__(mod)
    except Exception as e:
        print(f"  [_ver] import '{mod}' not available: {type(e).__name__}: {e}")
        return None
    try:
        return m.__version__
    except AttributeError:
        import importlib.metadata as ilm
        try:
            return ilm.version(mod)
        except Exception:
            return None

env_snapshot = {
    "notebook_id":      NOTEBOOK_ID,
    "date":             TODAY,
    "smoke":            bool(SMOKE),
    "python_version":   sys.version,
    "platform":         platform.platform(),
    "os_release":       platform.release(),
    "gpu":              _run(["nvidia-smi", "-L"]),
    "cuda":             _run(["nvcc", "--version"]),
    "git_commit":       _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "scgpt_commit":     SCGPT_COMMIT,
    "scgpt_version":    _ver("scgpt"),
    "peft_version":     _ver("peft"),
    "scanpy_version":   _ver("scanpy"),
    "anndata_version":  _ver("anndata"),
    "sklearn_version":  _ver("sklearn"),
    "torch_version":    _ver("torch"),
    "numpy_version":    _ver("numpy"),
    "model_checkpoint": os.path.basename(MODEL_DIR),
}
ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env{SUFFIX}.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

## 2 — The non-AD reference

### 2a — Load the reference and validate its schema

The forgetting probe needs an out-of-domain dataset with (i) **raw counts** (scGPT's input transform
starts from counts, exactly as the glia substrate did) and (ii) trustworthy **cell-type labels** to
classify. The reference config is isolated in one block so it can be swapped without touching
anything downstream.

**Default = the same PBMC file colab_13 used** (Kang et al. 2018, GSE96583), already on Drive from
that run; the download block is a fallback for a fresh Drive. Reusing it is deliberate and is the
whole basis of the cross-FM read: a different reference would confound "which model forgot" with
"which reference was used". The stricter breadth alternative (Tabula Sapiens) is a config swap, but
taking it would forfeit comparability with colab_13 unless that notebook were re-run too.

The asserts fail loud on missing raw counts or labels rather than silently embedding garbage.

In [ ]:
import gc
import numpy as np, pandas as pd, anndata as ad, scanpy as sc, scipy.sparse as sp

try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:28s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except ImportError:
    def _ram(tag): pass

sc.settings.verbosity = 1

# --- reference config (the one block to edit to swap references) ---
REF_H5AD         = os.path.join(DRIVE_ROOT, "reference", "pbmc_reference.h5ad")  # Kang 2018, GSE96583
REF_CELLTYPE_COL = "cell_type"     # obs column holding the cell-type labels to classify
REF_DONOR_COL    = "replicate"     # obs column for the donor-disjoint split (Kang 2018: 8 donors)
REF_RAW_LAYER    = None            # layer holding raw counts; None => use .X (or .raw if .X is normalized)
REF_NAME         = "pbmc"          # tag used in output filenames / audit
REF_SOURCE_URL   = "https://exampledata.scverse.org/pertpy/kang_2018.h5ad"

if not os.path.exists(REF_H5AD) and REF_NAME == "pbmc":
    print("reference not found on Drive -- downloading Kang et al. 2018 (GSE96583) ...")
    os.makedirs(os.path.dirname(REF_H5AD), exist_ok=True)
    import urllib.request, shutil
    tmp_path = REF_H5AD + ".part"
    # The host sits behind Cloudflare, which 403s urllib's default `Python-urllib/x.y` User-Agent
    # (verified 2026-07-21: Python-urllib -> 403; browser/curl/requests UAs -> 200).
    req = urllib.request.Request(REF_SOURCE_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(tmp_path, "wb") as fh:
        expected = resp.headers.get("Content-Length")
        shutil.copyfileobj(resp, fh)
    got = os.path.getsize(tmp_path)
    if expected is not None and got != int(expected):
        os.remove(tmp_path)   # never leave a truncated file a later run would treat as complete
        raise RuntimeError(f"truncated download: got {got} bytes, expected {expected}")
    os.rename(tmp_path, REF_H5AD)   # atomic: a crash mid-download leaves .part, never a short "real" file
    print(f"downloaded -> {REF_H5AD} ({got/1e6:.1f} MB)")

assert os.path.exists(REF_H5AD), (
    f"reference not found: {REF_H5AD}. Place the same labeled raw-counts non-AD h5ad colab_13 used "
    "there, or the cross-FM comparison this notebook is built around is not available.")

ref = sc.read_h5ad(REF_H5AD)
print("reference:", ref.shape)

# --- resolve raw counts ---
if REF_RAW_LAYER is not None:
    assert REF_RAW_LAYER in ref.layers, f"layer '{REF_RAW_LAYER}' absent; layers={list(ref.layers)}"
    ref.X = ref.layers[REF_RAW_LAYER].copy()

def _int_frac(X):
    idx = np.random.default_rng(0).choice(X.shape[0], size=min(2000, X.shape[0]), replace=False)
    d = X[idx]; d = d.data if sp.issparse(d) else np.asarray(d).ravel()
    return float(np.mean(np.mod(d, 1) == 0)) if d.size else 0.0

if _int_frac(ref.X) < 0.99 and ref.raw is not None:
    print("`.X` looks normalized -- falling back to `.raw` for raw counts")
    ref = ref.raw.to_adata()
assert _int_frac(ref.X) >= 0.99, (
    "reference `.X` is not raw counts -- scGPT's input transform (normalize_total + log1p + binning) "
    "must start from counts, as it did for the glia substrate")

# --- validate labels ---
assert REF_CELLTYPE_COL in ref.obs.columns, f"reference lacks cell-type column '{REF_CELLTYPE_COL}'"
ref.obs["cell_type"] = ref.obs[REF_CELLTYPE_COL].astype(str)
assert ref.obs["cell_type"].nunique() >= 5, (
    f"only {ref.obs['cell_type'].nunique()} cell types -- too few for a meaningful forgetting probe")
HAS_DONOR = REF_DONOR_COL is not None and REF_DONOR_COL in ref.obs.columns
ref.obs["donor_id"] = ref.obs[REF_DONOR_COL].astype(str) if HAS_DONOR else "single"
print(f"cell types: {ref.obs['cell_type'].nunique()} | donors: {ref.obs['donor_id'].nunique()} "
      f"(donor-disjoint split: {HAS_DONOR})")
print(ref.obs["cell_type"].value_counts().to_string())
_ram("reference loaded")

### 2b — Per-type cap, cross-FM cell-set identity check, scGPT vocabulary mapping

The cap, the seed and the iteration order are colab_13's, so this reproduces **the same 13,738 cells
across 8 types** that the Geneformer arm was scored on — and the assert below hard-stops if it does
not, because a silent cell-set difference would masquerade as an FM difference in §6. `ref_index` is
assigned after the `SMOKE` thinning so it numbers the rows actually embedded.

Gene mapping is where the two arms legitimately diverge: Geneformer needed symbol to Ensembl to
token, scGPT matches HGNC symbols against its own vocabulary directly. Coverage is reported rather
than gated on any niche gene — that hard-fail is glia and eval-#2 specific, and the forgetting probe
only needs broad coverage. One case is *not* benign and is checked explicitly: a cell left with zero
in-vocab genes cannot be embedded at all, and dropping it would break the cell-set identity the
assert above just established, so it is counted, reported, and recorded in the audit rather than
quietly removed.

In [ ]:
from scgpt.tokenizer import GeneVocab

# --- per-type cap: colab_13's values and iteration order, reproducing its exact cell set ---
rng = np.random.default_rng(SEED)
keep_idx = []
for ct_name, _ in ref.obs.groupby("cell_type", observed=True):
    pos = np.where(ref.obs["cell_type"].values == ct_name)[0]
    keep_idx.append(pos if len(pos) <= CAP_PER_TYPE
                    else rng.choice(pos, CAP_PER_TYPE, replace=False))
keep_idx = np.sort(np.concatenate(keep_idx))
ref = ref[keep_idx].copy()
print(f"after per-type cap ({CAP_PER_TYPE}): {ref.n_obs} cells across "
      f"{ref.obs['cell_type'].nunique()} types")

# --- hard stop on any drift from the cell set the Geneformer arm scored ---
GF_REF_CELLS = 13738     # geneformer_cpt_forgetting.n_ref_cells
GF_REF_TYPES = 8         # geneformer_cpt_forgetting.n_cell_types
if not SMOKE:
    assert ref.n_obs == GF_REF_CELLS, (
        f"capped reference is {ref.n_obs} cells, not colab_13's {GF_REF_CELLS} -- the two FM arms "
        "would be scored on different cells, so a cross-FM difference in section 6 could not be "
        "attributed to the models. Check the reference file, CAP_PER_TYPE and SEED before running.")
    assert ref.obs["cell_type"].nunique() == GF_REF_TYPES, (
        f"{ref.obs['cell_type'].nunique()} cell types, not colab_13's {GF_REF_TYPES}")
    print(f"cell-set identity with colab_13 OK: {GF_REF_CELLS} cells / {GF_REF_TYPES} types")

# --- SMOKE thinning: before embedding, so a plumbing run is actually cheap ---
# Thinned per (cell_type x donor) rather than per cell_type: the 5a split is donor-disjoint, so
# every donor and every type must survive or the split logic is not exercised at all. Placed ahead
# of ref_index, which is the realignment key every pass reindexes on.
if SMOKE:
    _rng = np.random.default_rng(SEED)
    _ctv, _dnv = ref.obs["cell_type"].values, ref.obs["donor_id"].values
    _keep = []
    for _c in pd.unique(_ctv):
        for _d in pd.unique(_dnv):
            _p = np.where((_ctv == _c) & (_dnv == _d))[0]
            if len(_p) == 0:
                continue
            _keep.append(_p if len(_p) <= SMOKE_N_PER_GROUP
                         else _rng.choice(_p, SMOKE_N_PER_GROUP, replace=False))
    ref = ref[np.sort(np.concatenate(_keep))].copy()
    print(f"[SMOKE] subsampled to {ref.n_obs} cells | types {ref.obs['cell_type'].nunique()} | "
          f"donors {ref.obs['donor_id'].nunique()}")

ref.obs["ref_index"] = np.arange(ref.n_obs)
ref.var["gene_name"] = ref.var_names

# --- scGPT vocabulary ---
VOCAB_FILE  = os.path.join(MODEL_DIR, "vocab.json")
CONFIG_FILE = os.path.join(MODEL_DIR, "args.json")
with open(CONFIG_FILE) as f:
    model_configs = json.load(f)

PAD_TOKEN = "<pad>"
vocab = GeneVocab.from_file(VOCAB_FILE)
for s in [PAD_TOKEN, "<cls>", "<eoc>"]:
    if s not in vocab:
        vocab.append_token(s)
print("\nscGPT vocabulary size:", len(vocab))

# scGPT's vocabulary is uppercase HGNC symbols; try the reference's symbols as given first and fall
# back to uppercased only if that is materially better, reporting which was used.
names_raw = ref.var["gene_name"].astype(str).tolist()
names_up  = [g.upper() for g in names_raw]
frac_raw  = float(np.mean([g in vocab for g in names_raw]))
frac_up   = float(np.mean([g in vocab for g in names_up]))
if frac_up > frac_raw + 0.01:
    print(f"using uppercased gene symbols ({frac_up:.1%} in vocab vs {frac_raw:.1%} as given)")
    ref.var["gene_name"] = names_up
    CASE_USED = "upper"
else:
    CASE_USED = "as_given"
ref.var["id_in_vocab"] = [vocab[g] if g in vocab else -1 for g in ref.var["gene_name"]]
n_vocab   = int((ref.var["id_in_vocab"] >= 0).sum())
FRAC_VOCAB = round(n_vocab / ref.n_vars, 4)
print(f"genes in the scGPT vocabulary: {n_vocab} / {ref.n_vars} ({FRAC_VOCAB:.1%}) "
      f"[symbols: {CASE_USED}]")
assert FRAC_VOCAB >= 0.30, (
    f"only {FRAC_VOCAB:.1%} of reference genes are in the scGPT vocabulary -- var_names may be "
    "Ensembl IDs rather than symbols, or the wrong ID type entirely")

# --- cells left with no in-vocab gene cannot be embedded at all ---
ref_v = ref[:, ref.var["id_in_vocab"] >= 0].copy()
_Xtmp = ref_v.X.tocsr() if sp.issparse(ref_v.X) else sp.csr_matrix(ref_v.X)
_Xtmp.eliminate_zeros()
N_ZERO_INVOCAB = int((np.diff(_Xtmp.indptr) == 0).sum())
CELLSET_EXACT  = (N_ZERO_INVOCAB == 0)
if not CELLSET_EXACT:
    print(f"\nWARNING: {N_ZERO_INVOCAB} cell(s) have no detected in-vocab gene and cannot be "
          "embedded -- dropping them. The scored cell set is then NOT identical to colab_13's, and "
          "the cross-FM comparison in section 6 is approximate by that many cells (recorded in the "
          "audit trace as cellset_exact=false).")
    _keep_rows = np.where(np.diff(_Xtmp.indptr) > 0)[0]
    ref   = ref[_keep_rows].copy()
    ref_v = ref_v[_keep_rows].copy()
    ref.obs["ref_index"]   = np.arange(ref.n_obs)
    ref_v.obs["ref_index"] = np.arange(ref_v.n_obs)
else:
    print("\nevery cell retains at least one in-vocab gene -- cell set is exactly colab_13's")
del _Xtmp; gc.collect()
print(f"in-vocab matrix: {ref_v.shape}")
_ram("after vocab mapping")

## 3 — Input geometry and the stochasticity budget

### 3a — Normalize, build the model inputs, and measure how much of the reference is over context

The input transform is the one every scGPT notebook in this project has used: `normalize_total(1e4)`
then `log1p` on raw counts, with the value binning left to the collator.

The number this section exists to produce is the **fraction of reference cells above the 1,200-token
context**, because that fraction *is* the stochasticity budget. Every cell above it gets a fresh
random gene subsample on every pass; every cell below it embeds deterministically. On the glia
substrate that fraction was 86.34% and the run was stochastic essentially everywhere. PBMC cells
carry far fewer detected genes than these snRNA glia, so the fraction here could be much smaller —
which would make the primary read nearly deterministic and the §5c deterministic-subset control
nearly the whole reference. Either way it is measured, not assumed, and it decides how much weight
the null in §5b has to carry.

The batch-geometry check is the standing guard from colab_13's crash: `batch x heads x max_len^2`
must stay under the int32 limit or attention overflows inside the kernel.

In [ ]:
import torch
from scgpt.model import TransformerModel
from scgpt.data_collator import DataCollator
from torch.utils.data import DataLoader, SequentialSampler

DEVICE = torch.device("cuda")

# PyTorch's attention modules carry an inference fast path that reads in_proj_weight as a raw tensor
# instead of calling the module's forward. Disabled explicitly, exactly as colab_16/17 did.
torch.backends.mha.set_fastpath_enabled(False)
print("attention fast path enabled:", torch.backends.mha.get_fastpath_enabled())

# --- input transform: identical to colab_10 3a / colab_16 3a / colab_17 2a ---
sc.pp.normalize_total(ref_v, target_sum=1e4)
sc.pp.log1p(ref_v)
print("applied normalize_total(1e4) + log1p")

vocab.set_default_index(vocab[PAD_TOKEN])
GENE_IDS  = np.array(vocab(ref_v.var["gene_name"].tolist()), dtype=int)
assert len(GENE_IDS) == ref_v.n_vars, "gene-id array is not aligned to the in-vocab panel"
CLS_ID    = vocab["<cls>"]
PAD_ID    = vocab[PAD_TOKEN]
PAD_VALUE = model_configs["pad_value"]
print(f"<cls> {CLS_ID} | <pad> {PAD_ID} | pad_value {PAD_VALUE}")

# --- input geometry and the stochasticity budget ---
Xv = ref_v.X
Xv = Xv.tocsr() if sp.issparse(Xv) else sp.csr_matrix(Xv)
Xv.eliminate_zeros()
nnz_per_cell = np.diff(Xv.indptr)
assert int((nnz_per_cell == 0).sum()) == 0, "a cell has no detected in-vocab gene (2b should have caught this)"

MAX_LENGTH = None   # adopted from colab_16's record at 4a; asserted against the value used here
MAX_LENGTH_ASSUMED = 1200
seq_len   = nnz_per_cell + 1                       # +1 for the prepended <cls> token
OVER_MASK = seq_len > MAX_LENGTH_ASSUMED           # re-checked against the recorded value at 4a
FRAC_OVER = float(OVER_MASK.mean())
print(f"\ndetected in-vocab genes per cell: median {int(np.median(nnz_per_cell))} | "
      f"p95 {int(np.percentile(nnz_per_cell, 95))} | max {int(nnz_per_cell.max())}")
print(f"cells above the {MAX_LENGTH_ASSUMED}-token context: {int(OVER_MASK.sum())} ({FRAC_OVER:.1%})")
print(f"  -> {FRAC_OVER:.1%} of cells are randomly gene-subsampled on every pass (stochastic);")
print(f"     {1-FRAC_OVER:.1%} embed deterministically and form the 5c control subset.")
print(f"  (glia substrate for comparison: 86.34% over context -- colab_16)")

INT32MAX  = 2**31 - 1
N_HEADS   = model_configs["nheads"]
EMB_BATCH = 64
while EMB_BATCH > 1 and EMB_BATCH * N_HEADS * MAX_LENGTH_ASSUMED**2 >= INT32MAX:
    EMB_BATCH //= 2
_elems = EMB_BATCH * N_HEADS * MAX_LENGTH_ASSUMED**2
assert _elems < INT32MAX, "attention score tensor exceeds the int32 limit even at batch 1"
print(f"\nbatch geometry OK: {EMB_BATCH} x {N_HEADS} heads x {MAX_LENGTH_ASSUMED}^2 = "
      f"{_elems:,} < {INT32MAX:,}")
_ram("inputs ready")

## 4 — Embedding passes

### 4a — Inventory the checkpoint, the adapter and colab_16's recorded numbers

Everything this probe needs about the checkpoint it is scoring is read from the recorded audit trail
rather than restated by hand: the adapter location, the embedding dimension, the context length, the
LoRA target configuration and the detector #1 magnitudes §6 reads the result against. The scGPT
commit is asserted equal to the one the CPT run was produced under — a different commit would mean
the adapter is being merged into a differently-built model.

`MAX_LENGTH` is adopted here from that record and checked against the value §3a computed its geometry
under, so the over-context fraction reported above is guaranteed to describe the passes actually
run.

In [ ]:
AUDIT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_PATH) as f:
    audit = json.load(f)

cpt = audit["scgpt_cpt_aggregated"]
assert cpt["status"] == "computed", "colab_16's scGPT CPT run is not recorded as computed"
assert cpt["fm"] == "scgpt" and cpt["regime"] == "aggregated", "unexpected CPT record"
assert cpt["scgpt_commit"] == SCGPT_COMMIT, (
    f"installed scGPT {SCGPT_COMMIT[:7]} != the commit the CPT checkpoint was produced under "
    f"{cpt['scgpt_commit'][:7]} -- the adapter would be merged into a differently-built model")

ADAPTER_DIR = os.path.join(DRIVE_ROOT, cpt["adapter_file"])
assert os.path.exists(ADAPTER_DIR), f"adapter not found on Drive: {ADAPTER_DIR} (colab_16 output)"
EMB_DIM     = cpt["emb_dim"]
MAX_LENGTH  = cpt["max_length"]
LORA_TARGETS_RECORDED = cpt["lora"]["targets"]
assert MAX_LENGTH == MAX_LENGTH_ASSUMED, (
    f"colab_16 recorded max_length={MAX_LENGTH} but section 3a measured the over-context fraction "
    f"and batch geometry at {MAX_LENGTH_ASSUMED} -- re-run 3a with the recorded value")
assert EMB_DIM == model_configs["embsize"], (
    f"recorded emb_dim {EMB_DIM} != checkpoint embsize {model_configs['embsize']}")

D1           = cpt["detector_1"]
FLOOR_D1     = D1["noise_floor_measured"]
DRIFT_ALL    = D1["drift_all"]
PCT_OF_REF   = cpt["drift_pct_of_substate_reference"]
assert not D1["inert"], "colab_16 recorded the CPT checkpoint as inert -- there is nothing to probe"

gf = audit["geneformer_cpt_forgetting"]
assert gf["status"] == "computed", "colab_13's Geneformer forgetting probe is not recorded as computed"
assert gf["reference"] == REF_NAME, (
    f"colab_13 used reference '{gf['reference']}' but this run uses '{REF_NAME}' -- the cross-FM "
    "comparison in section 6 would not be like-for-like")
GF_EVAL3 = gf["eval3_detector2"]

print("checkpoint under probe:", cpt["adapter_file"])
print(f"  emb_dim {EMB_DIM} | max_length {MAX_LENGTH} | LoRA targets {LORA_TARGETS_RECORDED}")
print(f"  detector #1: drift_all {DRIFT_ALL:.5f} = {D1['drift_over_floor']:.2f}x its measured floor "
      f"({FLOOR_D1:.5f})")
print(f"  drift as % of the within-donor substate reference: "
      f"micro {PCT_OF_REF['microglia']:.1f}% | astro {PCT_OF_REF['astrocyte']:.1f}%")
print("\nGeneformer arm to compare against (colab_13, same reference):")
for _lyr, _rec in GF_EVAL3.items():
    print(f"  {_lyr}: zero-shot {_rec['acc_zeroshot']:.4f} -> CPT {_rec['acc_cpt']:.4f} | "
          f"drop {_rec['drop_pp']:+.2f} pp | {_rec['eval3_verdict']} | {_rec['detector2_gate']}")

PASS_DIR = os.path.join(DRIVE_ROOT, "scgpt_forgetting")
os.makedirs(PASS_DIR, exist_ok=True)
BASE_PASS_PATHS = [os.path.join(PASS_DIR, f"ref_{REF_NAME}_base_p{i}{SUFFIX}.h5ad")
                   for i in range(1, N_BASE_PASSES + 1)]
CPT_PASS_PATHS  = [os.path.join(PASS_DIR, f"ref_{REF_NAME}_cpt_{RUN_TAG}_p{i}{SUFFIX}.h5ad")
                   for i in range(1, N_CPT_PASSES + 1)]
print(f"\npass files -> {PASS_DIR} ({len(BASE_PASS_PATHS)} base + {len(CPT_PASS_PATHS)} cpt)")

### 4b — Repeat passes: frozen base x 5, merged adapter x 5, with a determinism check

Both arms are built from the same released checkpoint with the same loader; the CPT arm then reloads
colab_16's adapter and merges it. Three structural checks run before any merged embedding is
produced, because each failure mode is silent: that the stored adapter's target configuration is the
one recorded for this checkpoint, that the adapter actually attached to `self_attn` / `linear1` /
`linear2` on a real encoder layer, and that it did **not** attach to the value encoder (the
contamination the regex targeting exists to prevent).

Passes are written to Drive one at a time and dropped from memory, and an existing file is reused
rather than recomputed, so an interrupted session resumes instead of restarting.

After the base passes, a **determinism check** compares two of them on the under-context cells only.
Those cells are never subsampled, so their embeddings should be identical across passes to floating
point. If they are not, then random gene subsampling is *not* the only source of pass-to-pass
variation and both the null in §5b and the control in §5c rest on a false premise — so the check
prints the observed maximum deviation rather than assuming it.

In [ ]:
from peft import PeftModel
import re
from tqdm.auto import tqdm

RENAME_RULES = {                      # the released checkpoint uses flash-attention naming
    r"self_attn\._impl\.Wqkv\.":     "self_attn.in_proj_",
    r"self_attn\.Wqkv\.":            "self_attn.in_proj_",
    r"self_attn\._impl\.out_proj\.": "self_attn.out_proj.",
}
REQUIRED_PREFIXES = ("encoder.", "value_encoder.", "transformer_encoder.", "decoder.")

def build_scgpt(model_configs, vocab):
    """Construct the model with the same arguments the library's embedding path uses."""
    return TransformerModel(
        ntoken=len(vocab), d_model=model_configs["embsize"], nhead=model_configs["nheads"],
        d_hid=model_configs["d_hid"], nlayers=model_configs["nlayers"],
        nlayers_cls=model_configs["n_layers_cls"], n_cls=1, vocab=vocab,
        dropout=model_configs["dropout"], pad_token=model_configs["pad_token"],
        pad_value=model_configs["pad_value"], do_mvc=True, do_dab=False,
        use_batch_labels=False, domain_spec_batchnorm=False, explicit_zero_prob=False,
        use_fast_transformer=False,     # no flash-attn -> PyTorch attention, as in every other pass
        fast_transformer_backend="flash", pre_norm=False,
    )

def load_checkpoint_verbose(model, ckpt_path):
    """Load the checkpoint and REPORT what was dropped -- the loader itself drops silently."""
    raw = torch.load(ckpt_path, map_location="cpu")
    renamed = {}
    for k, v in raw.items():
        kk = k
        for pat, rep in RENAME_RULES.items():
            kk = re.sub(pat, rep, kk)
        renamed[kk] = v
    model_dict = model.state_dict()
    kept    = {k: v for k, v in renamed.items() if k in model_dict and v.shape == model_dict[k].shape}
    dropped = sorted(set(renamed) - set(kept))
    model_dict.update(kept)
    model.load_state_dict(model_dict)
    return kept, dropped

def assert_required_loaded(model, kept, tag):
    """Every parameter on the input->readout path must have come from the checkpoint."""
    missing = [n for n, _ in model.named_parameters()
               if n.startswith(REQUIRED_PREFIXES) and n not in kept]
    assert not missing, (
        f"[{tag}] {len(missing)} parameter(s) on the input->readout path were NOT loaded from the "
        f"checkpoint, e.g. {missing[:8]}")
    print(f"  [{tag}] all required parameter groups loaded from the checkpoint")

class RefCellDataset(torch.utils.data.Dataset):
    """One cell -> (gene ids, expression values) with <cls> prepended, densified per row."""
    def __init__(self, X_csr, gene_ids, cls_id, pad_value):
        self.X, self.gene_ids, self.cls_id, self.pad_value = X_csr, gene_ids, cls_id, pad_value
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        row = self.X[idx].toarray().ravel()
        nz = np.nonzero(row)[0]
        genes  = np.insert(self.gene_ids[nz], 0, self.cls_id)
        values = np.insert(row[nz], 0, self.pad_value)
        return {"id": idx,
                "genes": torch.from_numpy(genes).long(),
                "expressions": torch.from_numpy(values).float()}

full_ds = RefCellDataset(Xv, GENE_IDS, CLS_ID, PAD_VALUE)
NUM_WORKERS = 2

def make_collator():
    # sampling=True draws the random gene subsample for over-context cells, and the collator runs
    # per BATCH -- so a fresh collator per pass genuinely redraws it. do_mlm=False: embedding path.
    return DataCollator(
        do_padding=True, pad_token_id=PAD_ID, pad_value=PAD_VALUE,
        do_mlm=False, do_binning=True, mask_value=-1, max_length=MAX_LENGTH,
        sampling=True, keep_first_n_tokens=1,
    )

@torch.no_grad()
def embed_cells(model, dataset, desc=""):
    """<cls>-position cell embeddings, L2-normalized -- the same readout as every stored matrix."""
    loader = DataLoader(dataset, batch_size=EMB_BATCH, sampler=SequentialSampler(dataset),
                        collate_fn=make_collator(), drop_last=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    model.eval()
    out = np.zeros((len(dataset), model_configs["embsize"]), dtype=np.float32)
    count = 0
    for batch in tqdm(loader, desc=desc or "embedding"):
        gene = batch["gene"].to(DEVICE, non_blocking=True)
        expr = batch["expr"].to(DEVICE, non_blocking=True)
        pad_mask = gene.eq(PAD_ID)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            h = model._encode(gene, expr, src_key_padding_mask=pad_mask, batch_labels=None)
        e = h[:, 0, :].float().cpu().numpy()          # <cls> position
        out[count:count + len(e)] = e
        count += len(e)
    assert count == len(dataset), f"embedded {count} of {len(dataset)} cells"
    return out / (np.linalg.norm(out, axis=1, keepdims=True) + 1e-12)

PASS_OBS = ["ref_index", "cell_type", "donor_id"]

def _save_pass(X, path):
    a = ad.AnnData(X=X, obs=ref.obs[PASS_OBS].copy())
    a.write_h5ad(path)
    print(f"  saved {os.path.basename(path)} ({os.path.getsize(path)/1e6:.0f} MB)")
    del a; gc.collect()

# ------------------------------------------------ frozen base
print(f"=== frozen base: {N_BASE_PASSES} passes (the null -- true effect is zero) ===")
if all(os.path.exists(p) for p in BASE_PASS_PATHS):
    print("  all base passes already on Drive -- nothing to embed")
else:
    base_model = build_scgpt(model_configs, vocab)
    _kb, _db = load_checkpoint_verbose(base_model, os.path.join(MODEL_DIR, "best_model.pt"))
    assert_required_loaded(base_model, _kb, "base reload")
    base_model.to(DEVICE)
    for i, p in enumerate(BASE_PASS_PATHS, start=1):
        if os.path.exists(p):
            print(f"  pass {i}: exists, skipping"); continue
        X = embed_cells(base_model, full_ds, desc=f"base pass {i}/{N_BASE_PASSES}")
        _save_pass(X, p); del X; gc.collect()
    del base_model; gc.collect(); torch.cuda.empty_cache()

# ------------------------------------------------ merged adapter
print(f"\n=== merged CPT adapter ({RUN_TAG}): {N_CPT_PASSES} passes ===")
if all(os.path.exists(p) for p in CPT_PASS_PATHS):
    print("  all CPT passes already on Drive -- nothing to embed")
else:
    cpt_model = build_scgpt(model_configs, vocab)
    _kc, _dc = load_checkpoint_verbose(cpt_model, os.path.join(MODEL_DIR, "best_model.pt"))
    assert_required_loaded(cpt_model, _kc, "CPT base reload")

    peft_reload = PeftModel.from_pretrained(cpt_model, ADAPTER_DIR)
    _cfg_targets = peft_reload.peft_config["default"].target_modules
    print("  adapter target_modules as stored:", _cfg_targets)
    # peft may hand this back as the bare regex string or wrapped in a set depending on how it
    # round-trips the config. Compare structurally rather than via a repr-substring match: str(a_set)
    # escapes backslashes in its repr, so a regex needle like `\.` would never match inside a set's
    # str() even when the set's actual (unescaped) member is identical to the needle.
    if isinstance(_cfg_targets, str):
        _targets_ok = _cfg_targets == LORA_TARGETS_RECORDED
    else:
        _targets_ok = LORA_TARGETS_RECORDED in set(_cfg_targets)
    assert _targets_ok, (
        f"the adapter on Drive targets {_cfg_targets!r}, but audit_report.json records "
        f"{LORA_TARGETS_RECORDED!r} for this checkpoint -- different LoRA target configuration")

    _l0 = cpt_model.transformer_encoder.layers[0]
    for nm, mod in (("self_attn", _l0.self_attn), ("linear1", _l0.linear1), ("linear2", _l0.linear2)):
        assert hasattr(mod, "lora_A"), (
            f"the reloaded adapter did not attach to {nm} on layer 0 -- peft raises already if "
            "nothing matched anywhere, so a gap here means something narrower is wrong (only some "
            "layers, or a differently-configured adapter); merging would silently hand back "
            "something other than a LoRA-carrying model")
    _leak = [n for n, m in cpt_model.named_modules()
             if n.startswith("value_encoder") and hasattr(m, "lora_A")]
    assert not _leak, f"the adapter attached to the value encoder as well: {_leak}"
    print("  adapter attached to encoder self_attn/linear1/linear2, value encoder untouched")

    merged = peft_reload.merge_and_unload().to(DEVICE)
    for i, p in enumerate(CPT_PASS_PATHS, start=1):
        if os.path.exists(p):
            print(f"  pass {i}: exists, skipping"); continue
        X = embed_cells(merged, full_ds, desc=f"cpt pass {i}/{N_CPT_PASSES}")
        _save_pass(X, p); del X; gc.collect()
    del cpt_model, peft_reload, merged; gc.collect(); torch.cuda.empty_cache()

assert all(os.path.exists(p) for p in BASE_PASS_PATHS + CPT_PASS_PATHS), \
    "passes are still missing after the embedding loop"

# ------------------------------------------------ load every pass, aligned on ref_index
def _load_pass(path):
    a = sc.read_h5ad(path)
    assert a.n_obs == ref.n_obs, f"{os.path.basename(path)} has {a.n_obs} rows, expected {ref.n_obs}"
    order = np.argsort(a.obs["ref_index"].to_numpy())
    assert np.array_equal(a.obs["ref_index"].to_numpy()[order], ref.obs["ref_index"].to_numpy()), \
        f"{os.path.basename(path)} does not carry this run's ref_index set"
    X = np.asarray(a.X, dtype=np.float32)[order]
    assert np.isfinite(X).all(), (
        f"{os.path.basename(path)} contains non-finite values (NaN/inf) -- likely a float16 "
        "autocast overflow during embedding; catching it here rather than letting it surface later "
        "as an opaque sklearn error")
    return X

BASE_EMB = [_load_pass(p) for p in BASE_PASS_PATHS]
CPT_EMB  = [_load_pass(p) for p in CPT_PASS_PATHS]
print(f"\nloaded {len(BASE_EMB)} base + {len(CPT_EMB)} cpt passes, each {BASE_EMB[0].shape}")

# ------------------------------------------------ determinism check on under-context cells
DET_MASK = ~OVER_MASK
n_det = int(DET_MASK.sum())
if n_det == 0:
    print("\ndeterminism check SKIPPED: no cell sits under the context limit")
    DET_MAXDIFF = None
else:
    DET_MAXDIFF = float(np.abs(BASE_EMB[0][DET_MASK] - BASE_EMB[1][DET_MASK]).max())
    print(f"\ndeterminism check on the {n_det} under-context cells (base pass 1 vs pass 2):")
    print(f"  max |difference| = {DET_MAXDIFF:.3e}")
    if DET_MAXDIFF > 1e-5:
        print("  WARNING: these cells are never gene-subsampled, so they should be identical across "
              "passes. A non-zero deviation means random subsampling is NOT the only source of "
              "pass-to-pass variation -- the null in 5b and the control in 5c both assume it is.")
    else:
        print("  as expected: under-context cells embed deterministically, so all pass-to-pass "
              "variation comes from gene subsampling of over-context cells")
_ram("passes loaded")

## 5 — Eval #3 / detector #2

### 5a — Reference split and scored-type set (must reproduce the Geneformer arm exactly)

The split is colab_13's: donor-disjoint when the reference carries more than three donors, otherwise
stratified by cell type. Because both arms are scored on the *same* split, any residual leakage
affects both equally and cancels in the drop.

Two floors, both inherited: types with fewer than 50 cells in aggregate are dropped before the split,
and types left with fewer than 20 cells in the held-out split are excluded from scoring afterwards. A
donor-disjoint split over eight donors cannot stratify by cell type, so a type can clear the first
floor and still be too thin to estimate. Balanced accuracy weights every class equally regardless of
size, so a thin class would inject noise into the drop that does **not** cancel between the arms —
exactly what a hard pass/fail gate should not be exposed to.

The scored-type set is then asserted equal to colab_13's. Cell-set identity was established in §2b;
this closes the loop by confirming the same *classes* are being scored, which is what makes the
cross-FM drops in §6 directly comparable numbers rather than two numbers about different problems.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit, GroupShuffleSplit

ct = ref.obs["cell_type"].to_numpy()
dn = ref.obs["donor_id"].to_numpy()

MIN_CELLS_PER_TYPE = 50
MIN_TEST_PER_TYPE  = 20

vc = pd.Series(ct).value_counts()
keep_types = set(vc[vc >= MIN_CELLS_PER_TYPE].index)
dropped_types = sorted(set(vc.index) - keep_types)
keep = np.isin(ct, list(keep_types))
print(f"cell types kept: {len(keep_types)} | dropped (<{MIN_CELLS_PER_TYPE} cells): {dropped_types}")

idx = np.where(keep)[0]
if ref.obs["donor_id"].nunique() > 3:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    tr_rel, te_rel = next(gss.split(idx, ct[idx], groups=dn[idx]))
    split_kind = "donor-disjoint"
else:
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    tr_rel, te_rel = next(sss.split(idx, ct[idx]))
    split_kind = "stratified-by-celltype"
train_idx, test_idx = idx[tr_rel], idx[te_rel]
print(f"split: {split_kind} | train {len(train_idx)} / test {len(test_idx)} cells")
print("test cell-type counts:")
print(pd.Series(ct[test_idx]).value_counts().to_string())

test_vc = pd.Series(ct[test_idx]).value_counts()
underpowered_types = sorted(test_vc[test_vc < MIN_TEST_PER_TYPE].index)
if underpowered_types:
    print(f"\nWARNING: {len(underpowered_types)} type(s) have <{MIN_TEST_PER_TYPE} test cells after "
          f"the split -- excluding from k-NN scoring: {underpowered_types}")
    train_idx = train_idx[~np.isin(ct[train_idx], underpowered_types)]
    test_idx  = test_idx[~np.isin(ct[test_idx],  underpowered_types)]
scored_types = sorted(keep_types - set(underpowered_types))
print(f"\nfinal: train {len(train_idx)} / test {len(test_idx)} cells, "
      f"{len(scored_types)} types scored: {scored_types}")

# --- the scored problem must be the one colab_13 scored, or section 6 compares two different tasks ---
GF_N_SCORED  = GF_EVAL3["L-1"]["n_types_scored"]
GF_EXCLUDED  = sorted(GF_EVAL3["L-1"]["excluded_types"])
if not SMOKE:
    assert len(scored_types) == GF_N_SCORED, (
        f"{len(scored_types)} types scored here vs colab_13's {GF_N_SCORED} -- the two arms would "
        "be scored on different classification problems")
    assert sorted(underpowered_types) == GF_EXCLUDED, (
        f"types excluded for thin test representation are {sorted(underpowered_types)}, but "
        f"colab_13 excluded {GF_EXCLUDED} -- the split or the reference differs")
    print(f"scored-problem identity with colab_13 OK: {GF_N_SCORED} types, excluding {GF_EXCLUDED}")

### 5b — k-NN cell-type accuracy per pass, the drop, and the bands against a measured null

For each pass a k-NN is fit on the train cells' embedding and scored by balanced accuracy on the
held-out cells — self-consistently, fitting and scoring within the same pass, which is how the
measurement would be made in practice. That gives five base accuracies and five CPT accuracies.

The headline **drop = mean(base) - mean(CPT)**, positive meaning forgetting, scored against eval #3's
bands and detector #2's gate exactly as `docs/EVALUATION_CONTRACT.md` states them.

The null it is read against is built from **within-arm** pass pairs — ten base-vs-base and ten
CPT-vs-CPT differences, twenty in total, every one of which has a true effect of zero by
construction. This is the widest null the run makes available, and it is built that way on purpose:
the recurring failure mode in this project has been a delta that clears a narrow null the code
happened to construct and then dissolves once checked against every same-model pair already on disk.
The worst-case cross-arm difference over all twenty-five base-vs-CPT pairings is printed alongside,
so the reported drop can be seen against its own spread rather than as a point estimate.

In [ ]:
import itertools
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score

K_NEIGHBORS = 15

def band_forget(drop_pp):        # eval #3 reporting bands (drop = base - cpt, positive = worse)
    if drop_pp > 15: return "catastrophic"
    if drop_pp > 5:  return "concerning"
    return "acceptable"

def gate_detector2(drop_pp):     # detector #2 hard gate
    return "OVERTRAIN" if drop_pp > 20 else "pass"

def knn_bacc(X, tr, te):
    sc_ = StandardScaler().fit(X[tr])
    knn = KNeighborsClassifier(n_neighbors=K_NEIGHBORS).fit(sc_.transform(X[tr]), ct[tr])
    return float(balanced_accuracy_score(ct[te], knn.predict(sc_.transform(X[te]))))

ACC_BASE = [knn_bacc(X, train_idx, test_idx) for X in BASE_EMB]
ACC_CPT  = [knn_bacc(X, train_idx, test_idx) for X in CPT_EMB]

print("per-pass balanced accuracy (k-NN, k=%d, %d types)" % (K_NEIGHBORS, len(scored_types)))
print("  base:", " ".join(f"{a:.4f}" for a in ACC_BASE),
      f"| mean {np.mean(ACC_BASE):.4f} sd {np.std(ACC_BASE, ddof=1):.4f}")
print("  cpt :", " ".join(f"{a:.4f}" for a in ACC_CPT),
      f"| mean {np.mean(ACC_CPT):.4f} sd {np.std(ACC_CPT, ddof=1):.4f}")

DROP_PP  = (np.mean(ACC_BASE) - np.mean(ACC_CPT)) * 100
VERDICT3 = band_forget(DROP_PP)
GATE2    = gate_detector2(DROP_PP)

# --- the null: every within-arm pair, where the true effect is zero by construction ---
NULL_BASE = [abs(a - b) * 100 for a, b in itertools.combinations(ACC_BASE, 2)]
NULL_CPT  = [abs(a - b) * 100 for a, b in itertools.combinations(ACC_CPT, 2)]
NULL_ALL  = NULL_BASE + NULL_CPT
NULL_MAX  = float(max(NULL_ALL))
NULL_MEAN = float(np.mean(NULL_ALL))
CROSS     = [(a - b) * 100 for a in ACC_BASE for b in ACC_CPT]

print(f"\ndrop (mean base - mean cpt): {DROP_PP:+.2f} pp")
print(f"  eval #3 band: {VERDICT3}   (acceptable <=5 / concerning 5-15 / catastrophic >15)")
print(f"  detector #2 gate: {GATE2}   (>20 pp drop = OVERTRAIN)")
print(f"\nmeasured null from {len(NULL_ALL)} within-arm pass pairs (true effect zero):")
print(f"  mean |delta| {NULL_MEAN:.2f} pp | max |delta| {NULL_MAX:.2f} pp")
print(f"  base-only max {max(NULL_BASE):.2f} pp | cpt-only max {max(NULL_CPT):.2f} pp")
print(f"cross-arm pairings (all {len(CROSS)}): {min(CROSS):+.2f} to {max(CROSS):+.2f} pp")
ABOVE_NULL = abs(DROP_PP) > NULL_MAX
print(f"\nis the drop distinguishable from re-embedding stochasticity? "
      f"|{DROP_PP:+.2f}| vs null max {NULL_MAX:.2f} -> {'YES' if ABOVE_NULL else 'NO'}")
print("  (the contract bands above are applied to the drop unchanged either way -- the measured "
      "null is reported alongside them, not substituted for them)")

### 5c — The deterministic subset, pass-to-pass stability by arm, and which read is operative

Two checks on whether the primary number can be trusted, plus the pre-registered rule for what to do
if it can't.

**Per-arm stability.** colab_17 found the merged adapter roughly half as sensitive to random gene
subsampling as the frozen base on the glia substrate. If that reproduces here, the two arms are being
measured under *different* amounts of embedding noise, and the quieter arm (CPT) is flattered: noise
degrades a k-NN, so a noisier base arm scores lower, which shrinks the drop and makes detector #2
anti-conservative — under-reporting forgetting is the one direction a safety gate must not fail in.
The magnitude is measured here so §6 can say how much room that leaves.

**The deterministic subset.** Cells under the context limit are never subsampled, so both arms embed
them identically on every pass and the asymmetry above cannot exist. Re-running the probe on that
subset alone gives a drop with no stochastic component at all. It is a biased sample — low-complexity
cells, and cell types are not equally represented among them, so it silently changes *what* is being
classified. The composition shift is printed rather than assumed, and the subset is only scored if it
still clears the same test-cell floor.

**Which read is operative.** The full-reference drop is the default for eval #3 / detector #2 *unless
both* of these hold: the deterministic-subset read is available (≥2 scored types clear the floor on
it), and the arms are measurably asymmetric (`STAB_RATIO < 0.80`, this cell's own threshold for
"CPT arm is quieter"). Only then does §6 draw the gate from the deterministic subset instead. This is
fixed here, before either drop is known, specifically so the choice can't be made by seeing which read
is more favorable — it fires only when the named bias is actually observed, not on the mere
possibility of it. The full-reference number is still computed, printed, and logged either way; what
changes is only which one the pass/fail decision and the value written back to colab_17 are drawn
from. Cost accepted if it fires: the gate is then read off a narrower, low-complexity-skewed type set
rather than the full scored problem — traded for immunity to a bias that is, by construction, worse
for a safety gate than that skew is.

In [ ]:
import math

# ---------------------------------------------------------------- per-arm pass-to-pass stability
def mean_pair_cosdist(mats, mask):
    """Median per-cell cosine distance between passes on `mask` cells, averaged over pass pairs.
    Restricted to stochastically-subsampled (over-context) cells: on a deterministic cell, repeat
    passes are bitwise identical, so 1-cos(theta) is pure float32 summation noise (randomly signed
    around zero) rather than a real instability measurement. Including deterministic cells would
    make the ratio meaningless, not just noisier, whenever they are the majority of the reference."""
    vals = []
    for A, B in itertools.combinations(mats, 2):
        vals.append(float(np.median(1.0 - np.sum(A[mask] * B[mask], axis=1))))
    return float(np.mean(vals)), vals

MIN_STAB_CELLS = 50   # below this the median is too noisy to trust as a promotion trigger
n_over = int(OVER_MASK.sum())
if n_over < MIN_STAB_CELLS:
    STAB_BASE = STAB_CPT = STAB_RATIO = float("nan")
    print(f"pass-to-pass stability SKIPPED: only {n_over} over-context cells (need "
          f">={MIN_STAB_CELLS}) -- too few for the median to be a trustworthy promotion signal. "
          "Promotion cannot fire on this reference's stochasticity asymmetry.")
else:
    STAB_BASE, _sb = mean_pair_cosdist(BASE_EMB, OVER_MASK)
    STAB_CPT,  _sc = mean_pair_cosdist(CPT_EMB, OVER_MASK)
    STAB_RATIO = (STAB_CPT / STAB_BASE) if STAB_BASE > 0 else float("nan")
    print(f"pass-to-pass embedding instability, over-context cells only (n={n_over}, median "
          "per-cell cosine distance between passes):")
    print(f"  base {STAB_BASE:.5f} | cpt {STAB_CPT:.5f} | ratio cpt/base {STAB_RATIO:.2f}")
    print("  colab_17 on the glia substrate: 0.00363 base vs 0.0015 merged (ratio ca. 0.41)")
    if STAB_RATIO < 0.8:
        print("  -> the CPT arm is measurably QUIETER, so its k-NN is measured under less embedding "
              "noise than the base arm's. Direction of the bias: it shrinks the drop, i.e. it makes "
              "detector #2 anti-conservative. The deterministic-subset read below is the control.")
    elif STAB_RATIO > 1.25:
        print("  -> the CPT arm is NOISIER than the base arm, which biases the drop upward "
              "(over-reporting forgetting rather than under-reporting it)")
    else:
        print("  -> the two arms are comparably stable; the asymmetry colab_17 saw does not carry to "
              "this reference, and the primary read needs no stochastic correction")

# ---------------------------------------------------------------- deterministic subset
det_train = train_idx[DET_MASK[train_idx]]
det_test  = test_idx[DET_MASK[test_idx]]
det_vc    = pd.Series(ct[det_test]).value_counts()
det_thin  = sorted(det_vc[det_vc < MIN_TEST_PER_TYPE].index)
det_missing = sorted(set(scored_types) - set(det_vc.index))

print(f"\ndeterministic subset (cells under the {MAX_LENGTH}-token context): "
      f"{int(DET_MASK.sum())} of {len(DET_MASK)} cells")
print(f"  train {len(det_train)} / test {len(det_test)}")
print("  composition shift, share of each type retained in the test split:")
full_vc = pd.Series(ct[test_idx]).value_counts()
for t in scored_types:
    kept_t = int(det_vc.get(t, 0)); tot_t = int(full_vc.get(t, 0))
    print(f"    {t:<28s} {kept_t:>5d} / {tot_t:<5d} ({(kept_t/tot_t if tot_t else 0):.0%})")

DET_SCORED = sorted(set(scored_types) - set(det_thin) - set(det_missing))
DET_FULL_COVERAGE = set(DET_SCORED) == set(scored_types)
_dt_candidate = det_train[np.isin(ct[det_train], DET_SCORED)] if DET_SCORED else det_train[:0]
if len(DET_SCORED) < 2 or len(_dt_candidate) < K_NEIGHBORS:
    print(f"\ndeterministic-subset read UNAVAILABLE: {len(DET_SCORED)} type(s) clear the "
          f"{MIN_TEST_PER_TYPE}-test-cell floor, {len(_dt_candidate)} train cells available "
          f"(k-NN needs >={K_NEIGHBORS})")
    DET_RESULT = None
else:
    if det_thin or det_missing:
        print(f"\nNOTE: {sorted(set(det_thin) | set(det_missing))} fall below the "
              f"{MIN_TEST_PER_TYPE}-test-cell floor on this subset and are excluded from it -- so "
              "this read covers a NARROWER set of types than the primary one, on top of the "
              "composition shift above")
    _dt = _dt_candidate
    _de = det_test[np.isin(ct[det_test],  DET_SCORED)]
    det_acc_base = knn_bacc(BASE_EMB[0], _dt, _de)
    det_acc_cpt  = knn_bacc(CPT_EMB[0],  _dt, _de)
    det_drop     = (det_acc_base - det_acc_cpt) * 100
    # every pass is identical on these cells, so one pass per arm is the whole measurement --
    # verified rather than asserted, on BOTH arms:
    _chk_base = (max(float(np.abs(BASE_EMB[0][_de] - X[_de]).max()) for X in BASE_EMB[1:])
                 if len(BASE_EMB) > 1 else None)
    _chk_cpt  = (max(float(np.abs(CPT_EMB[0][_de] - X[_de]).max()) for X in CPT_EMB[1:])
                 if len(CPT_EMB) > 1 else None)
    DET_RESULT = {"n_types": len(DET_SCORED), "types": DET_SCORED,
                  "full_coverage": DET_FULL_COVERAGE,
                  "n_train": int(len(_dt)), "n_test": int(len(_de)),
                  "acc_base": round(det_acc_base, 4), "acc_cpt": round(det_acc_cpt, 4),
                  "drop_pp": round(det_drop, 2), "eval3_verdict": band_forget(det_drop),
                  "detector2_gate": gate_detector2(det_drop),
                  "max_pass_deviation_base": _chk_base, "max_pass_deviation_cpt": _chk_cpt}
    _cov_label = "full coverage, same problem as the primary read" if DET_FULL_COVERAGE else \
        "PARTIAL coverage -- a narrower/easier problem than the primary read, not eligible to " \
        "become operative regardless of the stability read"
    print(f"\ndeterministic subset, {len(DET_SCORED)} of {len(scored_types)} types ({_cov_label}), "
          f"{len(_de)} test cells:")
    print(f"  base {det_acc_base:.4f} -> cpt {det_acc_cpt:.4f} | drop {det_drop:+.2f} pp | "
          f"{band_forget(det_drop)} | {gate_detector2(det_drop)}")
    print(f"  max deviation across passes on these cells: base "
          f"{'n/a (1 pass)' if _chk_base is None else f'{_chk_base:.3e}'} | cpt "
          f"{'n/a (1 pass)' if _chk_cpt is None else f'{_chk_cpt:.3e}'} (should be ca. 0 -- one pass "
          "per arm is the entire measurement here)")
    print(f"  primary read for comparison: {DROP_PP:+.2f} pp over {len(scored_types)} types")

## 6 — Summary, cross-FM comparison, gate, handoff

### 6a — Verdict, the gate applied back to colab_17, the cross-FM read, audit trace, commit commands

Four things close here.

**Which read is operative.** Before anything else, the promotion rule decides whether the gate is
drawn from the full reference or the deterministic subset. Promotion needs *all three*: the subset
available, `STAB_RATIO < 0.80` (and actually measured — see 5c's `MIN_STAB_CELLS` floor), and the
subset scoring the exact same type set as the primary read (`DET_FULL_COVERAGE`). That last condition
is what stops a promotion from quietly swapping in an easier, smaller-class problem that would shrink
the drop for reasons unrelated to the stochastic asymmetry it exists to correct for. Everything below
labeled "the gate" uses that operative value; the cross-FM row is the one deliberate exception, held
to the full reference for comparability.

**The gate.** Detector #2's verdict is applied back to colab_17's recorded evals. colab_17 wrote its
results with `detector_2_gate: not_run_for_scgpt` and flagged any meaningful/decisive verdict as a
candidate win pending exactly this probe; this cell rewrites that field with the operative outcome, so
the scGPT arm's audit trail no longer carries an open dependency.

**The cross-FM read.** Geneformer and scGPT are now scored on the same reference, the same cells, the
same classes and the same k-NN, which is what the contract's "report both FMs" clause is for. This row
always uses the full-reference numbers, even when the gate above used the deterministic subset instead
— otherwise a promoted scGPT read would be compared against a Geneformer number computed on a
different, larger set of cells, which is exactly the mismatch this section exists to avoid. The
comparison is stated with the one asymmetry it carries: Geneformer was read at two extraction points
because damage could hide below its readout, scGPT at one that sits above every adapted parameter.

**The standing question.** colab_16 measured a drift that exceeds the distance between biologically
distinct cell states; colab_17 showed it lands on neither scored axis. Whether it also spared general
cell-type knowledge is what decides how the whole scGPT arc reads, and this cell prints that composite
conclusion — from the operative values — rather than leaving it to prose.

In [ ]:
import shlex

print("=== EVAL #3 / DETECTOR #2 (forgetting on the non-AD %s reference) ===" % REF_NAME)
print(f"cells {ref.n_obs} | types scored {len(scored_types)} | split {split_kind} "
      f"(train {len(train_idx)} / test {len(test_idx)})")
print(f"base   mean {np.mean(ACC_BASE):.4f} (sd {np.std(ACC_BASE, ddof=1):.4f}, "
      f"{len(ACC_BASE)} passes)")
print(f"cpt    mean {np.mean(ACC_CPT):.4f} (sd {np.std(ACC_CPT, ddof=1):.4f}, "
      f"{len(ACC_CPT)} passes)")
print(f"drop (full reference) {DROP_PP:+.2f} pp -> eval #3: {VERDICT3} | detector #2: {GATE2}")
print(f"       measured null (within-arm pairs) max {NULL_MAX:.2f} pp -> drop is "
      f"{'above' if ABOVE_NULL else 'within'} it")

# ---------------------------------------------------------------- operative-read selection
# Pre-registered here, before either drop is inspected: the full-reference read is operative for
# the gate unless ALL THREE hold: the deterministic-subset read is available, it scores the exact
# same type set as the primary read (DET_FULL_COVERAGE -- otherwise promotion could swap in an
# easier, smaller-class problem that shrinks the drop for reasons unrelated to the asymmetry below),
# and the arms are measurably asymmetric in embedding stability (STAB_RATIO < 0.8, 5c's own
# threshold for "CPT arm is quieter", computed only when 5c had enough over-context cells to trust
# it -- STAB_RATIO is nan otherwise, and nan < 0.8 is False, so promotion correctly cannot fire).
# All three conditions are orthogonal to the drop values themselves, so this is not chosen by
# looking at which read is more favorable.
DET_ELIGIBLE = (
    DET_RESULT is not None
    and DET_RESULT["full_coverage"]
    and STAB_RATIO < 0.8
)
if DET_ELIGIBLE:
    OPERATIVE_SOURCE  = "deterministic_subset"
    OPERATIVE_DROP    = DET_RESULT["drop_pp"]
    OPERATIVE_VERDICT = DET_RESULT["eval3_verdict"]
    OPERATIVE_GATE    = DET_RESULT["detector2_gate"]
else:
    OPERATIVE_SOURCE  = "full_reference"
    OPERATIVE_DROP    = DROP_PP
    OPERATIVE_VERDICT = VERDICT3
    OPERATIVE_GATE    = GATE2

print("\n=== OPERATIVE READ FOR DETECTOR #2 ===")
print(f"  promotion-eligible: {DET_ELIGIBLE} (subset available: {DET_RESULT is not None}, "
      f"full coverage: {DET_RESULT['full_coverage'] if DET_RESULT else 'n/a'}, "
      f"stab_ratio {STAB_RATIO:.2f} < 0.80: {STAB_RATIO < 0.8})")
print(f"  operative source: {OPERATIVE_SOURCE} | drop {OPERATIVE_DROP:+.2f} pp | "
      f"{OPERATIVE_VERDICT} | {OPERATIVE_GATE}")
if DET_RESULT:
    _det_label = "OPERATIVE" if DET_ELIGIBLE else "corroborating, not operative"
    _mpd_base = DET_RESULT["max_pass_deviation_base"]
    _mpd_cpt  = DET_RESULT["max_pass_deviation_cpt"]
    _mpd_base_s = "n/a" if _mpd_base is None else f"{_mpd_base:.3e}"
    _mpd_cpt_s  = "n/a" if _mpd_cpt  is None else f"{_mpd_cpt:.3e}"
    print(f"  deterministic subset ({_det_label}, {DET_RESULT['n_types']} types, "
          f"full_coverage={DET_RESULT['full_coverage']}): "
          f"{DET_RESULT['drop_pp']:+.2f} pp -> {DET_RESULT['eval3_verdict']} | "
          f"{DET_RESULT['detector2_gate']} | max pass deviation base {_mpd_base_s}, "
          f"cpt {_mpd_cpt_s} "
          "(null ~0 by construction, no re-embedding noise on these cells)")
else:
    print("  deterministic subset: UNAVAILABLE (too few scored types or train cells clear the "
          "floors)")

# ---------------------------------------------------------------- gate applied back to colab_17
print("\n=== DETECTOR #2 GATE APPLIED TO colab_17 ===")
# Hard requirement, not a soft .get(): colab_17 must have already run and committed its audit trace,
# or this notebook cannot honestly claim to have applied anything to it. An absence here should stop
# the run, not fall through to a print claiming there was nothing to gate.
assert "scgpt_cpt_evals" in audit, (
    "audit_report.json has no scgpt_cpt_evals record -- colab_17 must run and commit its audit "
    "trace before this gate can be applied. Check AUDIT_PATH and confirm colab_17 wrote that key.")
ev = audit["scgpt_cpt_evals"]
gated_wins = []
VERDICT_VOCAB = {"noise", "meaningful", "decisive", "regression", "underpowered"}
seen_verdicts = []
for eval_key in ("eval1_substate_probe", "eval2_apoe_recovery"):
    for name, rec in (ev.get(eval_key) or {}).items():
        for k, v in (rec.items() if isinstance(rec, dict) else []):
            if isinstance(v, str) and v in VERDICT_VOCAB:
                seen_verdicts.append(f"{eval_key}.{name}.{k}={v}")
                if v in ("meaningful", "decisive"):
                    gated_wins.append(f"{eval_key}.{name}.{k}={v}")
# "no wins to gate" is only meaningful if the scan could see the verdicts in the first place.
# Without this, a change in how colab_17 nests its records would make the gate silently blind
# and report "nothing to gate" -- an absence claim that cannot fail on its own.
assert seen_verdicts, (
    "found no verdict field anywhere in colab_17's eval records -- the gate cannot confirm "
    "there is nothing to overturn, so it is not being applied. Check the structure of "
    "audit_report.json['scgpt_cpt_evals'] before reading the result below.")
print(f"  scanned {len(seen_verdicts)} recorded verdict(s) in colab_17's evals")
if not gated_wins:
    print("  colab_17 recorded no meaningful/decisive verdict, so this gate overturns nothing.")
    print("  It still closes the open dependency: the scGPT arm's null evals are now null under "
          "a run whose forgetting status is measured rather than unknown.")
elif OPERATIVE_GATE == "OVERTRAIN":
    print(f"  detector #2 TRIPPED (source: {OPERATIVE_SOURCE}) -- the following are logged as "
          "overtrain-confounded, not wins:")
    for w in gated_wins: print("   ", w)
else:
    print(f"  detector #2 passed (source: {OPERATIVE_SOURCE}) -- these verdicts stand as wins: "
          f"{gated_wins}")

# ---------------------------------------------------------------- cross-FM
# Always the full-reference read here, regardless of which source is operative for the gate --
# this comparison's whole point is "same reference, same cells, same classes" vs Geneformer, and
# Geneformer's colab_13 numbers are on the full reference too.
print("\n=== CROSS-FM (same reference, same cells, same classes, same k-NN) ===")
for _lyr, _rec in GF_EVAL3.items():
    print(f"  Geneformer {_lyr:<4s}: {_rec['acc_zeroshot']:.4f} -> {_rec['acc_cpt']:.4f} | "
          f"drop {_rec['drop_pp']:+.2f} pp | {_rec['eval3_verdict']} | {_rec['detector2_gate']}")
print(f"  scGPT <cls> : {np.mean(ACC_BASE):.4f} -> {np.mean(ACC_CPT):.4f} | "
      f"drop {DROP_PP:+.2f} pp | {VERDICT3} | {GATE2}")
print("  asymmetry to keep in view: Geneformer needed two extraction points because its adapted "
      "parameters sit downstream of the pipeline readout; scGPT's single <cls> readout sits above "
      "every adapted parameter, so one number is the complete view there.")
if DET_ELIGIBLE:
    print(f"  note: the gate applied above used the deterministic-subset read instead of this "
          "full-reference one (promotion triggered); this row stays on the full reference so it "
          "remains comparable to Geneformer's.")

# ---------------------------------------------------------------- the composite read
print("\n=== WHERE THIS LEAVES THE scGPT ARC ===")
print(f"  detector #1 (colab_16): drift REAL, {D1['drift_over_floor']:.2f}x its floor, "
      f"{PCT_OF_REF['microglia']:.1f}% / {PCT_OF_REF['astrocyte']:.1f}% of the substate reference")
print(f"  evals #1/#2 (colab_17): {'no meaningful/decisive verdict' if not gated_wins else gated_wins}")
print(f"  eval #3 (here, {OPERATIVE_SOURCE}): {OPERATIVE_VERDICT}, drop {OPERATIVE_DROP:+.2f} pp, "
      f"detector #2 {OPERATIVE_GATE}")
if OPERATIVE_GATE == "pass" and OPERATIVE_VERDICT == "acceptable" and not gated_wins:
    print("  -> a large, real representational movement that shows up on neither scored biological "
          "axis AND has not measurably damaged general cell-type knowledge. On everything this "
          "project can currently measure, the movement is orthogonal to all of it.")

# ---------------------------------------------------------------- audit trace
if SMOKE:
    print("\nSMOKE run -- audit trace NOT written")
else:
    _stab_base_json  = round(STAB_BASE, 6)  if math.isfinite(STAB_BASE)  else None
    _stab_cpt_json   = round(STAB_CPT, 6)   if math.isfinite(STAB_CPT)   else None
    _stab_ratio_json = round(STAB_RATIO, 3) if math.isfinite(STAB_RATIO) else None

    audit["scgpt_cpt_forgetting"] = {
        "status": "computed",
        "date": TODAY,
        "fm": "scgpt",
        "reads_run": "scgpt_cpt_aggregated",
        "run_tag": RUN_TAG,
        "scgpt_commit": SCGPT_COMMIT,
        "reference": REF_NAME,
        "reference_file": os.path.relpath(REF_H5AD, DRIVE_ROOT),
        "n_ref_cells": int(ref.n_obs),
        "n_cell_types": int(pd.Series(ct).nunique()),
        "cellset_exact_vs_geneformer": bool(CELLSET_EXACT),
        "n_cells_no_invocab_gene": int(N_ZERO_INVOCAB),
        "vocab_audit": {"frac_in_vocab": FRAC_VOCAB, "symbol_case": CASE_USED},
        "extraction_point": {
            "readout": "<cls> at the top of the encoder",
            "complete": True,
            "note": ("every LoRA target is upstream of the readout and the ExprDecoder is frozen "
                     "downstream, so a single extraction point sees the complete adapted "
                     "representation -- unlike the Geneformer arm, which needed L-1 and L0")},
        "split": {"kind": split_kind, "seed": SEED, "test_size": 0.30,
                  "n_train": int(len(train_idx)), "n_test": int(len(test_idx)),
                  "min_cells_per_type": MIN_CELLS_PER_TYPE,
                  "min_test_per_type": MIN_TEST_PER_TYPE,
                  "types_scored": scored_types,
                  "types_excluded_thin_test": sorted(underpowered_types),
                  "types_dropped_rare": dropped_types},
        "eval3_detector2": {
            "k_neighbors": K_NEIGHBORS,
            "acc_base_passes": [round(a, 4) for a in ACC_BASE],
            "acc_cpt_passes":  [round(a, 4) for a in ACC_CPT],
            "acc_base_mean": round(float(np.mean(ACC_BASE)), 4),
            "acc_cpt_mean":  round(float(np.mean(ACC_CPT)), 4),
            "drop_pp": round(float(DROP_PP), 2),
            "eval3_verdict": VERDICT3,
            "detector2_gate": GATE2,
            "n_types_scored": len(scored_types),
            "excluded_types": sorted(underpowered_types)},
        "measured_null": {
            "n_base_passes": len(ACC_BASE), "n_cpt_passes": len(ACC_CPT),
            "within_arm_pairs": len(NULL_ALL),
            "null_mean_abs_pp": round(NULL_MEAN, 3),
            "null_max_abs_pp": round(NULL_MAX, 3),
            "cross_arm_range_pp": [round(min(CROSS), 3), round(max(CROSS), 3)],
            "drop_above_null": bool(ABOVE_NULL),
            "note": ("the null is every within-arm pass pair, where the true effect is zero by "
                     "construction -- the widest null this run makes available. Contract bands were "
                     "applied to the drop unchanged; the null is reported alongside, not "
                     "substituted for them.")},
        "stochasticity": {
            "frac_cells_over_context": round(FRAC_OVER, 4),
            "max_length": MAX_LENGTH,
            "determinism_check_max_dev": DET_MAXDIFF,
            "n_cells_used_for_stability": n_over,
            "pass_instability_base": _stab_base_json,
            "pass_instability_cpt": _stab_cpt_json,
            "pass_instability_ratio_cpt_over_base": _stab_ratio_json,
            "note": ("measured on over-context cells only (see 5c); null/not-a-number when there "
                     "were too few of them to trust the median. If the CPT arm is quieter than the "
                     "base arm, its k-NN is measured under less embedding noise, which SHRINKS the "
                     "drop and makes detector #2 anti-conservative. The deterministic-subset read "
                     "is the control for it.")},
        "deterministic_subset": DET_RESULT,
        "operative_gate": {
            "source": OPERATIVE_SOURCE,
            "eligible": bool(DET_ELIGIBLE),
            "drop_pp": round(float(OPERATIVE_DROP), 2),
            "eval3_verdict": OPERATIVE_VERDICT,
            "detector2_gate": OPERATIVE_GATE,
            "note": ("the value applied to colab_17's detector_2_gate field below and used in the "
                     "composite scGPT-arc conclusion above; eval3_detector2 above always records "
                     "the full-reference read regardless, for cross-FM comparability. Promoted to "
                     "the deterministic subset only when it is available, scores the same type set "
                     "as the primary read, AND stab_ratio < 0.80 -- conditions fixed before either "
                     "drop was inspected.")},
        "detector2_primary_gate": OPERATIVE_GATE,
        "cross_fm": {
            "geneformer": GF_EVAL3,
            "note": ("same reference, cap, seed, split and k-NN settings; cell-set identity asserted "
                     "at 2b and scored-class identity at 5a. Geneformer read at two extraction "
                     "points, scGPT at one that is complete by construction.")},
        "note": ("eval #3 and detector #2 are one k-NN read two ways. PBMC's myeloid overlap with "
                 "microglia may under-detect forgetting in that lineage; the lymphoid types carry "
                 "the out-of-domain part of the probe. Tabula Sapiens is the stricter breadth swap "
                 "but would forfeit comparability with colab_13 unless that notebook were re-run."),
    }
    ev["detector_2_gate"] = {
        "status": "run",
        "source": "scgpt_cpt_forgetting",
        "date": TODAY,
        "drop_pp": round(float(OPERATIVE_DROP), 2),
        "eval3_verdict": OPERATIVE_VERDICT,
        "gate": OPERATIVE_GATE,
        "gated_verdicts": gated_wins,
        "note": ("colab_17 recorded this as not_run_for_scgpt; the forgetting probe has now run "
                 "on the same checkpoint and this field records its outcome (operative source: "
                 f"{OPERATIVE_SOURCE}).")}
    with open(AUDIT_PATH, "w") as f:
        json.dump(audit, f, indent=2)
    print("\naudit trace appended (and colab_17's detector_2_gate rewritten) ->", AUDIT_PATH)

    rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_PATH)]
    print("\n=== Commit + push (from WSL -- Colab has no git creds) ===")
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git add "
          + " ".join(shlex.quote(r) for r in rel))
    print("  # ALSO stage the downloaded executed notebook itself:")
    print("  #   git add notebooks/executed/colab_18_scgpt_cpt_forgetting_OUTPUT.ipynb")
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m "
          "'colab_18: scGPT CPT eval #3 / detector #2 (forgetting probe on a non-AD reference)'")
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git push")